### Дообучение модели на данных из TMDB

In [1]:
import os
import sys
import re
import json

from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from datasets import Dataset as DatasetForTransformers
from datasets import ClassLabel, Features, Value

load_dotenv()

c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Возьмем данные об отзывах с сайта TMDB из бд

In [2]:
sys.path.append('C:/Users/retro/ILYA/pet-projects/review-sentiment-analysis')

DATABASE_URL = os.getenv('DATABASE_URL') 

engine = create_engine(DATABASE_URL)

SessionLocal = sessionmaker(bind=engine)

db = SessionLocal()

In [3]:
from app.models import Review

reviews = db.query(Review).all()

review_texts = []
labels = []
for item in reviews:
    text = item.content.replace('<br />', ' ').strip()
    text = text.replace('\r', '').replace('\n', '').replace("\\'", "'")
    text = re.sub(r'<.*?>', '', text)

    if item.rating >= 7:
        labels.append(1)
        review_texts.append(text)
        
    elif item.rating <= 4:
        labels.append(0)
        review_texts.append(text)

In [4]:
with open('../data/test_data.json', 'r', encoding='utf-8') as file:
    test_data = json.load(file)

X_test = test_data['X']
y_test = test_data['y']

ds_train_val = DatasetForTransformers.from_dict({'text': review_texts, 'labels': labels})
ds_test = DatasetForTransformers.from_dict({'text': X_test, 'labels': y_test})

features = Features({
    'text': Value('string'),
    'labels': ClassLabel(names=['neg', 'pos'])
})

ds_train_val = ds_train_val.cast(features)
split = ds_train_val.train_test_split(test_size=0.1, seed=42, stratify_by_column='labels')
ds_train = split['train']
ds_val = split['test']

Casting the dataset: 100%|██████████| 97/97 [00:00<00:00, 228309.48 examples/s]


In [5]:
import numpy as np
import random
from datetime import datetime

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

import torch

from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

set_seed = seed

Дообучим модель с теми же параметрами обучения на новых данных полученных с TMDB

In [6]:
MODEL_SPECS = [
    {'name': '../best_model', 'max_len': 256, 'train_bs': 16, 'eval_bs': 32, 'lr': 2e-5, 'epochs': 3}
]

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

results_summary = []

for spec in MODEL_SPECS:
    model_name = spec['name']

    run_stamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    run_dir = f'../bert_runs/{model_name.replace('/', '_')}_{run_stamp}'

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(batch):
        return tokenizer(
            batch['text'],
            truncation=True,
            max_length=spec['max_len']
        )
    
    train_tok = ds_train.map(tokenize_fn, batched=True, remove_columns=['text'])
    val_tok = ds_val.map(tokenize_fn, batched=True, remove_columns=['text'])
    test_tok = ds_test.map(tokenize_fn, batched=True, remove_columns=['text'])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    args = TrainingArguments(
        output_dir=run_dir,
        learning_rate=spec['lr'],
        per_device_train_batch_size=spec['train_bs'],
        per_device_eval_batch_size=spec['eval_bs'],
        num_train_epochs=spec['epochs'],
        weight_decay=0.01,
        
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,

        logging_steps=50,
        report_to='none',
        fp16=True,
        max_grad_norm=1.0,
        optim='adamw_torch'
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # --- VAL: classification_report ---
    val_pred = trainer.predict(val_tok)
    val_preds = np.argmax(val_pred.predictions, axis=1)
    print('\nVAL classification_report:') 
    print(classification_report(ds_val['labels'], val_preds, target_names=['neg', 'pos'], digits=4))

    # --- TEST: classification_report --- 
    test_pred = trainer.predict(test_tok) 
    test_preds = np.argmax(test_pred.predictions, axis=1) 
    print('\nTEST classification_report:') 
    print(classification_report(ds_test['labels'], test_preds, target_names=['neg', 'pos'], digits=4))

    report_dict = classification_report(
        ds_test['labels'],
        test_preds,
        target_names=['neg', 'pos'],
        output_dict=True
    )

    # Сохраним краткий итог
    best_val = trainer.state.best_metric
    results_summary.append({
        'model': model_name,
        'run_dir': run_dir,
        'best_val_f1': float(best_val) if best_val is not None else None,
        'test_accuracy': float(accuracy_score(ds_test['labels'], test_preds)),
        'test_report': report_dict
    })

print('\n' + '='*80)
print('SUMMARY:')
for r in sorted(results_summary, key=lambda d: (d['best_val_f1'] is not None, d['best_val_f1']), reverse=True):
    print(r)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 700.86it/s, Materializing param=electra.encoder.layer.11.output.dense.weight]              


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
checkpoint_path = '../bert_runs/.._best_model_2026-03-17_10-31-42/checkpoint-18'

tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='eval_tmp',
        per_device_eval_batch_size=32,
        report_to='none'
    ),
    data_collator=data_collator,
)

pred = trainer.predict(test_tok)
test_preds = np.argmax(pred.predictions, axis=1)

print(classification_report(
    ds_test['labels'],
    test_preds,
    target_names=['neg', 'pos'],
    digits=4
))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 583.76it/s, Materializing param=electra.encoder.layer.11.output.dense.weight]              


KeyboardInterrupt: 

: 